# Jazan test: buildings, elevation, and Sentinel-2 discovery

This notebook repeats the Getting Started workflow for Jazan city:

- **Part A:** Overture building footprints inside a small Jazan area
- **Part B:** Copernicus DEM tiles for the same area
- **Part C:** zonal statistics (mean elevation per building) and a map
- **Part D:** Sentinel-2 discovery (prints the table and band names so the NDVI step can be written correctly)

Run the cells in order with **Shift + Enter**. A Micro runtime is enough.
**Stop the runtime when you finish.**

Note: Copernicus DEM is 30 m, which is coarse for single buildings. Treat Part C as a workflow test, not a final elevation product.

In [ ]:
from sedona.spark import *
import os

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## Area of interest

A small box over Jazan city, so the test stays fast and cheap. Change the four numbers to move or enlarge it.

In [ ]:
XMIN, YMIN, XMAX, YMAX = 42.50, 16.85, 42.65, 16.95   # lon/lat, EPSG:4326

AOI_WKT = (f"POLYGON(({XMIN} {YMIN}, {XMAX} {YMIN}, {XMAX} {YMAX}, "
           f"{XMIN} {YMAX}, {XMIN} {YMIN}))")
CENTER = {"lat": (YMIN + YMAX) / 2, "lng": (XMIN + XMAX) / 2, "zoom": 13}
OUT_DIR = os.getenv("USER_S3_PATH")

print("AOI:", AOI_WKT)
print("Output folder:", OUT_DIR)

## Helpers and table discovery

These functions find tables and the geometry or raster column automatically, so the notebook does not depend on exact names. The cell prints every table in the three databases used below.

In [ ]:
CATALOG = "wherobots_open_data"

def list_tables(db):
    rows = sedona.sql(f"SHOW TABLES IN {CATALOG}.{db}").collect()
    return [r["tableName"] for r in rows]

def col_kind(field):
    t = type(field.dataType).__name__.lower()
    try:
        t += " " + field.dataType.simpleString().lower()
    except Exception:
        pass
    if "raster" in t:
        return "raster"
    if "geometry" in t:
        return "geometry"
    return "other"

def find_col(df, kind):
    for f in df.schema.fields:
        if col_kind(f) == kind:
            return f.name
    return None

def pick_table(db, keywords):
    tables = list_tables(db)
    for kw in keywords:
        for t in tables:
            if kw in t.lower():
                return t
    return tables[0] if tables else None

for db in ["overture_maps_foundation", "copernicus_dem", "sentinel2"]:
    print(f"\n=== {db} ===")
    try:
        for t in list_tables(db):
            print("  ", t)
    except Exception as e:
        print("   could not list:", str(e)[:300])

## Part A: Overture buildings in Jazan

In [ ]:
bld_table = pick_table("overture_maps_foundation", ["buildings_building", "building"])
bld_full = f"{CATALOG}.overture_maps_foundation.{bld_table}"
bld = sedona.table(bld_full)
bld_geom = find_col(bld, "geometry")
print("Buildings table:", bld_full, "| geometry column:", bld_geom)

aoi_filter = f"ST_Intersects({bld_geom}, ST_GeomFromWKT('{AOI_WKT}'))"

# Overture's bbox column lets the engine skip files outside Jazan quickly
if "bbox" in bld.columns and "xmin" in bld.schema["bbox"].dataType.simpleString():
    aoi_filter = (f"bbox.xmin <= {XMAX} AND bbox.xmax >= {XMIN} AND "
                  f"bbox.ymin <= {YMAX} AND bbox.ymax >= {YMIN} AND " + aoi_filter)

select_cols = ["id AS bid" if "id" in bld.columns else "monotonically_increasing_id() AS bid",
               f"{bld_geom} AS geom"]
if "height" in bld.columns:
    select_cols.append("height")

jazan_bld = bld.where(aoi_filter).selectExpr(*select_cols).cache()
print("Buildings in the Jazan AOI:", jazan_bld.count())
jazan_bld.show(5, truncate=60)

## Part B: Copernicus DEM tiles over Jazan

In [ ]:
dem_table = pick_table("copernicus_dem", ["30", "glo"])
dem_full = f"{CATALOG}.copernicus_dem.{dem_table}"
dem = sedona.table(dem_full)
dem_rast = find_col(dem, "raster")
print("DEM table:", dem_full, "| raster column:", dem_rast)
dem.printSchema()

if dem_rast is None:
    raise ValueError("No raster column found in the DEM table. Send the schema above to Claude.")

jazan_dem = (dem.where(f"RS_Intersects({dem_rast}, ST_GeomFromWKT('{AOI_WKT}'))")
                .selectExpr(f"{dem_rast} AS rast")
                .cache())
n_tiles = jazan_dem.count()
print("DEM tiles touching the AOI:", n_tiles)

dem_srid = jazan_dem.selectExpr("RS_SRID(rast) AS s").first()["s"] if n_tiles else None
print("DEM CRS (EPSG):", dem_srid)
jazan_dem.selectExpr("RS_Metadata(rast) AS meta").show(3, truncate=False)

## Part C: mean elevation per building (zonal statistics)

Same pattern as the NYC Getting Started notebook: join buildings to the DEM tiles they touch, run `RS_ZonalStats`, and average across tiles for buildings that cross a tile edge.

In [ ]:
jazan_bld.createOrReplaceTempView("bldg")
jazan_dem.createOrReplaceTempView("dem")

if dem_srid and dem_srid != 4326:
    zone = f"ST_Transform(ST_SetSRID(b.geom, 4326), 'EPSG:4326', 'EPSG:{dem_srid}')"
else:
    zone = "b.geom"

bld_elev = sedona.sql(f"""
    SELECT b.bid, b.geom,
           AVG(RS_ZonalStats(d.rast, {zone}, 1, 'mean', true)) AS elevation
    FROM bldg b
    JOIN dem d ON RS_Intersects(d.rast, b.geom)
    GROUP BY b.bid, b.geom
""").where("elevation IS NOT NULL").cache()

print("Buildings with an elevation value:", bld_elev.count())
bld_elev.selectExpr("min(elevation)", "avg(elevation)", "max(elevation)").show()

## Map the result

In [ ]:
from wherobots_gl import Map

out_uri = OUT_DIR + "jazan_buildings_elevation.parquet"
bld_elev.write.format("geoparquet").mode("overwrite").save(out_uri)

e_min, e_max = bld_elev.selectExpr("min(elevation)", "max(elevation)").first()

Map(
    layers=[{
        "type": "geoparquet",
        "source": out_uri,
        "name": "Jazan buildings, mean elevation (m)",
        "colorByColumn": "elevation",
        "colorByColormap": {"type": "preset", "preset": "viridis"},
        "colorByDomain": [float(e_min), float(e_max)],
        "opacity": 0.85,
    }],
    view=CENTER,
    basemap="dark",
)

## Part D: Sentinel-2 discovery

This prints each Sentinel-2 table's schema and a few rows over Jazan. **Copy the output of this cell (and any errors) and send it to Claude**, so the NDVI step can be written with the real band and date column names.

In [ ]:
s2_tables = list_tables("sentinel2")
print("Sentinel-2 tables:", s2_tables)

for t in s2_tables:
    full = f"{CATALOG}.sentinel2.{t}"
    df = sedona.table(full)
    print(f"\n=== {full} ===")
    df.printSchema()

    plain = [f.name for f in df.schema.fields if col_kind(f) == "other"]
    geom = find_col(df, "geometry")
    rast = find_col(df, "raster")
    if geom:
        sample = df.where(f"ST_Intersects({geom}, ST_GeomFromWKT('{AOI_WKT}'))")
        print("Rows over Jazan (footprint filter):")
    elif rast:
        sample = df.where(f"RS_Intersects({rast}, ST_GeomFromWKT('{AOI_WKT}'))")
        print("Rows over Jazan (raster filter):")
    else:
        sample = df
        print("Sample rows (no spatial column to filter on):")

    if plain:
        sample.select(*plain[:12]).show(5, truncate=40)

## Done

1. Press **Cmd + S** to save.
2. Send the Part D output to Claude.
3. Go back to the **Code** page and stop the runtime (trash icon under Notebook Runtimes).